# Kortex — Code Language Model
### Trained from scratch on TPU v5e-8

A ~350M parameter decoder-only transformer trained on Python code.

**Setup:** Settings > Accelerator > TPU v5e-8

In [ ]:
!pip install datasets tokenizers transformers torch_xla

In [ ]:
import os, sys, json, time, math, gc
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    import torch_xla.utils.serialization as xser
    DEVICE = xm.xla_device()
    print(f"Using TPU: {DEVICE}")
    print(f"XLA devices: {xm.get_xla_supported_devices()}")
except Exception as e:
    DEVICE = torch.device("cpu")
    print(f"TPU not available, using CPU: {e}")

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
WRITE_TOKEN = user_secrets.get_secret("WRITE_TOKEN")
print("Secrets loaded.")

In [ ]:
CONFIG = {
    'vocab_size': 50257,
    'n_layers': 24,
    'n_heads': 16,
    'n_kv_heads': 4,
    'd_model': 1024,
    'd_ff': 2816,
    'max_seq_len': 1024,
    'dropout': 0.1,
    'norm_eps': 1e-6,
    'rope_theta': 10000.0,
    'batch_size': 4,
    'grad_accum': 16,
    'lr': 3e-4,
    'weight_decay': 0.1,
    'warmup_steps': 500,
    'max_steps': 30000,
    'min_lr_ratio': 0.1,
    'max_grad_norm': 1.0,
    'checkpoint_every': 500,
    'eval_every': 250,
    'eval_steps': 50,
    'log_every': 10,
    'max_time_hours': 8.5,
    'save_top_k': 3,
}
print(f"Config loaded. Model: ~{CONFIG['n_layers']}L x {CONFIG['d_model']}D")

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).type_as(x) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq_len)
    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
    def forward(self, x, seq_len):
        if seq_len > self.cos_cached.shape[0]:
            self._build_cache(seq_len)
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos, sin = cos[None, None, :, :], sin[None, None, :, :]
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout=0.1):
        super().__init__()
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
    def repeat_kv(self, x, n_rep):
        B, T, N, D = x.shape
        if n_rep == 1: return x
        return x[:, :, None, :, :].expand(B, T, n_rep, N, D).reshape(B, T, N*n_rep, D)
    def forward(self, x, cos, sin, mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        k, v = self.repeat_kv(k, self.n_rep), self.repeat_kv(v, self.n_rep)
        scale = 1.0 / math.sqrt(self.head_dim)
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        if mask is not None:
            attn = attn.masked_fill(mask[:, :, :T, :T] == 0, float("-inf"))
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_drop(self.o_proj(out))

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, dropout=0.1, eps=1e-6):
        super().__init__()
        self.norm1 = RMSNorm(d_model, eps)
        self.attn = GQAAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, eps)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        x = x + self.ffn(self.norm2(x))
        return x

class KortexModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['d_model'])
        self.drop = nn.Dropout(cfg['dropout'])
        self.rotary = RotaryEmbedding(cfg['d_model']//cfg['n_heads'], cfg['max_seq_len'], cfg['rope_theta'])
        self.layers = nn.ModuleList([TransformerBlock(cfg['d_model'], cfg['n_heads'], cfg['n_kv_heads'], cfg['d_ff'], cfg['dropout'], cfg['norm_eps']) for _ in range(cfg['n_layers'])])
        self.norm = RMSNorm(cfg['d_model'], cfg['norm_eps'])
        self.lm_head = nn.Linear(cfg['d_model'], cfg['vocab_size'], bias=False)
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"KortexModel: {n_params/1e6:.1f}M parameters")
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: torch.nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            torch.nn.init.normal_(m.weight, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rotary(x, T)
        mask = torch.tril(torch.ones(T, T, device=idx.device, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)
        for layer in self.layers:
            x = layer(x, cos, sin, mask)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss
    def generate(self, idx, max_new_tokens=256, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg['max_seq_len']:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("Model architecture defined.")

In [ ]:
from datasets import load_dataset

MAX_BUF = 500000

class CodeStreamBuffer:
    def __init__(self, seq_len=1024):
        self.seq_len = seq_len
        self.buffer = []
        self.samples_seen = 0
    def fill(self, iterator, tokenizer, max_samples=MAX_BUF):
        count = 0
        for item in iterator:
            if count >= max_samples: break
            code = item.get('code', '') or item.get('text', '')
            if len(code) < 50: continue
            code = code[:5000]
            ids = tokenizer.encode(code)
            self.buffer.extend(ids)
            self.samples_seen += 1
            count += 1
            if count % 10000 == 0:
                print(f"  Loaded {count} samples, buffer size: {len(self.buffer)} tokens")
                xm.mark_step() if hasattr(DEVICE, 'type') and DEVICE.type == 'xla' else None
        print(f"  Total: {count} samples, buffer: {len(self.buffer)} tokens")
    def get_batch(self, batch_size):
        needed = (self.seq_len + 1) * batch_size
        if len(self.buffer) < needed:
            return None
        xs, ys = [], []
        for _ in range(batch_size):
            idx = torch.randint(0, max(1, len(self.buffer) - self.seq_len - 1), (1,)).item()
            chunk = self.buffer[idx:idx + self.seq_len + 1]
            if len(chunk) < self.seq_len + 1: continue
            xs.append(chunk[:-1])
            ys.append(chunk[1:])
        if len(xs) < batch_size: return None
        self.buffer = self.buffer[batch_size * self.seq_len // 2:]
        return torch.tensor(xs, dtype=torch.long), torch.tensor(ys, dtype=torch.long)

print("Dataset utilities loaded.")

In [ ]:
class CosineScheduler:
    def __init__(self, optimizer, warmup, max_steps, min_lr=0.1):
        self.opt, self.warmup, self.max_steps = optimizer, warmup, max_steps
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.min_lr, self.step_n = min_lr, 0
    def step(self):
        self.step_n += 1
        if self.step_n < self.warmup:
            s = self.step_n / max(1, self.warmup)
        else:
            p = (self.step_n - self.warmup) / max(1, self.max_steps - self.warmup)
            s = self.min_lr + 0.5 * (1.0 - self.min_lr) * (1.0 + math.cos(math.pi * p))
        for pg, bl in zip(self.opt.param_groups, self.base_lrs):
            pg['lr'] = bl * s
    def get_lr(self):
        return [pg['lr'] for pg in self.opt.param_groups]

print("Scheduler loaded.")

In [ ]:
import requests
import subprocess

def push_github(token, msg="Update checkpoint"):
    try:
        subprocess.run(["git", "add", "-A"], capture_output=True, timeout=30)
        r = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, timeout=30)
        if not r.stdout.strip():
            print("  No changes.")
            return
        subprocess.run(["git", "commit", "-m", msg], capture_output=True, timeout=30)
        r = subprocess.run(["git", "push"], capture_output=True, text=True, timeout=60)
        if r.returncode == 0:
            print("  Pushed to GitHub.")
        else:
            print(f"  Push failed: {r.stderr[:200]}")
    except Exception as e:
        print(f"  GitHub error: {e}")

def push_hf(local_path, token, repo_name="kortex"):
    try:
        from huggingface_hub import HfApi, create_repo
        api = HfApi(token=token)
        repo_id = f"{repo_name}-model"
        try:
            create_repo(repo_id, token=token, exist_ok=True)
        except: pass
        api.upload_folder(folder_path=local_path, repo_id=repo_id, token=token)
        print(f"  Uploaded to HF: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"  HF error: {e}")

print("Push utilities loaded.")

In [ ]:
print("="*60)
print("Initializing Kortex training...")
print("="*60)

model = KortexModel(CONFIG).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.95),
    eps=1e-8,
)

scheduler = CosineScheduler(optimizer, CONFIG['warmup_steps'], CONFIG['max_steps'], CONFIG['min_lr_ratio'])

global_step = 0
best_val_loss = float('inf')
checkpoints = []
start_time = time.time()
MAX_SECONDS = CONFIG['max_time_hours'] * 3600

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/kortex-output', exist_ok=True)

print(f"Model on device: {DEVICE}")
print(f"Max time budget: {CONFIG['max_time_hours']}h")
print(f"Max steps: {CONFIG['max_steps']}")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

print("Training BPE tokenizer on code...")
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
trainer = BpeTrainer(vocab_size=CONFIG['vocab_size'], min_frequency=2, special_tokens=["[PAD]", "[BOS]", "[EOS]", "[UNK]"], show_progress=True)

ds = load_dataset('codeparrot/github-code', split='train', streaming=True, trust_remote_code=True)
train_files = []
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    count = 0
    for item in ds:
        code = item.get('code', '') or item.get('text', '')
        if len(code) > 50:
            f.write(code[:5000] + '\n')
            count += 1
        if count >= 100000: break
    train_files.append(f.name)

tokenizer.train(train_files, trainer)
TOKENIZER_PATH = '/kaggle/working/kortex-output/tokenizer.json'
tokenizer.save(TOKENIZER_PATH)
print(f"Tokenizer trained: vocab_size={tokenizer.get_vocab_size()}")
del ds; gc.collect()

In [ ]:
def encode(text):
    return tokenizer.encode(text).ids

print("Loading training data into buffer...")
ds = load_dataset('codeparrot/github-code', split='train', streaming=True, trust_remote_code=True)
buffer = CodeStreamBuffer(seq_len=CONFIG['max_seq_len'])
buffer.fill(iter(ds), encode, max_samples=MAX_BUF)
print(f"Buffer ready: {len(buffer.buffer)} tokens from {buffer.samples_seen} samples")
del ds; gc.collect()

In [ ]:
def save_ckpt(step, val_loss=None, is_best=False):
    global best_val_loss, checkpoints
    ckpt = {
        'step': step,
        'model_state': model.state_dict(),
        'optim_state': optimizer.state_dict(),
        'global_step': global_step,
        'best_val_loss': best_val_loss,
    }
    if is_best:
        p = '/kaggle/working/checkpoints/best.pt'
        xser.save(ckpt, p) if hasattr(DEVICE, 'type') and DEVICE.type == 'xla' else torch.save(ckpt, p)
        print(f"  [BEST] step={step} loss={val_loss:.4f}")
    p = f'/kaggle/working/checkpoints/step_{step}.pt'
    xser.save(ckpt, p) if hasattr(DEVICE, 'type') and DEVICE.type == 'xla' else torch.save(ckpt, p)
    checkpoints.append((step, p, val_loss or float('inf')))
    print(f"  Saved checkpoint step={step}")
    if len(checkpoints) > CONFIG['save_top_k'] + 1:
        checkpoints.sort(key=lambda x: x[2])
        while len(checkpoints) > CONFIG['save_top_k']:
            s, path, _ = checkpoints.pop(0)
            if os.path.exists(path): os.remove(path)
    push_github(WRITE_TOKEN, f"Checkpoint step {step}")

def load_ckpt():
    global global_step, best_val_loss
    best_p = '/kaggle/working/checkpoints/best.pt'
    if os.path.exists(best_p):
        ckpt = torch.load(best_p, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optim_state'])
        global_step = ckpt.get('global_step', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"  Resumed from step {global_step}")
        return True
    print("  No checkpoint found, starting fresh.")
    return False

print("Checkpoint utils loaded.")

In [ ]:
load_ckpt()
print(f"\nStarting training...")
print(f"Budget: {CONFIG['max_time_hours']}h | Max steps: {CONFIG['max_steps']}")
print(f"Batch: {CONFIG['batch_size']} x {CONFIG['grad_accum']} grad_accum = {CONFIG['batch_size']*CONFIG['grad_accum']} effective")
print("="*60)

grad_accum_count = 0
train_losses = []

while global_step < CONFIG['max_steps']:
    elapsed = time.time() - start_time
    if elapsed >= MAX_SECONDS:
        print(f"\nTIME LIMIT REACHED ({CONFIG['max_time_hours']}h). Stopping.")
        break

    remaining_h = (MAX_SECONDS - elapsed) / 3600
    if remaining_h < 0.5:
        print(f"\nLess than 30min remaining. Final save and stop.")
        break

    batch = buffer.get_batch(CONFIG['batch_size'])
    if batch is None:
        print("Refilling buffer...")
        try:
            ds = load_dataset('codeparrot/github-code', split='train', streaming=True, trust_remote_code=True)
            buffer.fill(iter(ds), encode, max_samples=MAX_BUF // 2)
            del ds; gc.collect()
        except Exception as e:
            print(f"  Refill error: {e}")
            time.sleep(10)
        continue

    x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)

    with torch.amp.autocast(device_type='xla', dtype=torch.bfloat16, enabled=True):
        _, loss = model(x, targets=y)
    loss.backward()
    grad_accum_count += 1

    if grad_accum_count % CONFIG['grad_accum'] == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
        global_step += 1
        xm.mark_step() if hasattr(DEVICE, 'type') and DEVICE.type == 'xla' else None

        train_losses.append(loss.item())

        if global_step % CONFIG['log_every'] == 0:
            avg_loss = sum(train_losses[-CONFIG['log_every']:]) / len(train_losses[-CONFIG['log_every']:])
            lr = scheduler.get_lr()[0]
            rem = (MAX_SECONDS - (time.time() - start_time)) / 3600
            print(f"Step {global_step:>6d} | loss={avg_loss:.4f} | lr={lr:.2e} | {rem:.1f}h remaining")

        if global_step % CONFIG['eval_every'] == 0:
            model.eval()
            eval_loss = 0
            eval_count = 0
            with torch.no_grad():
                for _ in range(CONFIG['eval_steps']):
                    eb = buffer.get_batch(CONFIG['batch_size'])
                    if eb is None: break
                    ex, ey = eb[0].to(DEVICE), eb[1].to(DEVICE)
                    with torch.amp.autocast(device_type='xla', dtype=torch.bfloat16, enabled=True):
                        _, el = model(ex, targets=ey)
                    eval_loss += el.item()
                    eval_count += 1
            if eval_count > 0:
                avg_eval = eval_loss / eval_count
                is_best = avg_eval < best_val_loss
                if is_best:
                    best_val_loss = avg_eval
                print(f"  EVAL step={global_step} loss={avg_eval:.4f} {'[BEST]' if is_best else ''}")
                save_ckpt(global_step, avg_eval, is_best)

        if global_step % CONFIG['checkpoint_every'] == 0 and global_step % CONFIG['eval_every'] != 0:
            save_ckpt(global_step)

print("\n" + "="*60)
print(f"Training complete. Steps: {global_step} | Best val loss: {best_val_loss:.4f}")
print(f"Total time: {(time.time()-start_time)/3600:.2f}h")
print("="*60)

In [ ]:
print("Saving final model...")
final_path = '/kaggle/working/kortex-output'
os.makedirs(final_path, exist_ok=True)

final_ckpt = {
    'model_state': model.state_dict(),
    'config': CONFIG,
    'global_step': global_step,
    'best_val_loss': best_val_loss,
}
xser.save(final_ckpt, f'{final_path}/kortex_final.pt') if hasattr(DEVICE, 'type') and DEVICE.type == 'xla' else torch.save(final_ckpt, f'{final_path}/kortex_final.pt')

import shutil
shutil.copy(TOKENIZER_PATH, f'{final_path}/tokenizer.json')

with open(f'{final_path}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Final model saved to {final_path}")
push_github(WRITE_TOKEN, "Final model save")
push_hf(final_path, HF_TOKEN)
print("Done!")

In [ ]:
print("\n" + "="*60)
print("Testing Kortex - generating code...")
print("="*60)

model.eval()
prompts = [
    "def fibonacci(n):\n",
    "class BinaryTree:\n    def __init__(self, val):\n",
    "import numpy as np\n\ndef matrix_multiply(a, b):\n",
    "def merge_sort(arr):\n    if len(arr) <= 1:\n",
]

for prompt in prompts:
    print(f"\nPrompt: {prompt.strip()}")
    print("-" * 40)
    input_ids = encode(prompt)
    x = torch.tensor([input_ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        out = model.generate(x, max_new_tokens=200, temperature=0.8, top_k=50)
    generated = tokenizer.decode(out[0].cpu().tolist())
    print(generated)
    print()